# Local Qwen/vLLM workflow

This template serves a vision-language model through vLLM's OpenAI-compatible API, then runs PaperMiner against it. Adapt the model, device, dtype, tensor parallelism, and context length to your hardware.

In [ ]:
%env PM_DB=papers.db
%env PM_QUERY=lithium solid electrolyte
%env PM_RECIPE=sse
%env PM_LOCAL_MODEL=Qwen/Qwen3-VL-30B-A3B-Instruct
%env PM_BASE_URL=http://127.0.0.1:8000/v1
%env PM_MAX_MODEL_LEN=120000
%env PM_OUTPUT=temp_qwen_materials.csv
%env PM_FINAL=qwen_materials.csv

## Start and verify the server

Run this only in an accelerator allocation with a compatible vLLM installation. The context length must fit the available memory.

In [ ]:
import os
import subprocess
import time

import requests

log = open("vllm.log", "w", encoding="utf-8")
server = subprocess.Popen(
    [
        "vllm", "serve", os.environ["PM_LOCAL_MODEL"],
        "--host", "127.0.0.1", "--port", "8000",
        "--max-model-len", os.environ["PM_MAX_MODEL_LEN"],
    ],
    stdout=log, stderr=subprocess.STDOUT,
)
for _ in range(120):
    try:
        response = requests.get("http://127.0.0.1:8000/v1/models", timeout=5)
        response.raise_for_status()
        break
    except requests.RequestException:
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not become ready; inspect vllm.log")

## Configure PaperMiner

The local provider does not require a hosted-model API key. Both profiles point at the same server because this Qwen model accepts text and image input.

In [ ]:
%%bash
set -euo pipefail
pm_model_config text --provider local --model "$PM_LOCAL_MODEL" --base-url "$PM_BASE_URL" --input-token-limit "$PM_MAX_MODEL_LEN"
pm_model_config vision --provider local --model "$PM_LOCAL_MODEL" --base-url "$PM_BASE_URL" --input-token-limit "$PM_MAX_MODEL_LEN"
pm_model_status

## Build, scrape, and store

Corpus discovery and downloading do not use the local model. Keep those stages off scarce accelerator resources when possible.

In [ ]:
%%bash
set -euo pipefail
pm_search "$PM_QUERY" "$PM_DB" --source openalex --count 25
pm_download "$PM_DB" --format both
pm_scrape "$PM_DB" "$PM_RECIPE" --mode text-images --image-context paper-text --count 5 --output "$PM_OUTPUT"
pm_store "$PM_DB" "$PM_OUTPUT" "$PM_FINAL" "$PM_RECIPE" --assume-yes